# UdaPlay 02 — Agent

**Author: Sam Sepassi**

This notebook wires the three tools (`retrieve_game`,
`evaluate_retrieval`, `game_web_search`) into a stateful agent and runs
it against several example queries.

State machine:

```
START -> RETRIEVE -> EVALUATE -> (REPORT | WEB_SEARCH -> REPORT) -> DONE
```

When evaluation says retrieval is insufficient, the agent falls back to
Tavily web search **and** writes the new findings back into the vector
store as long-term memory.


## 1. Setup


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from lib.vector_store import VectorStoreManager
from lib.agent import UdaPlayAgent


## 2. Load the persistent vector store

We don't re-ingest here — that was notebook 01's job. We just open the
existing collection.


In [ ]:
store = VectorStoreManager(persist_directory='chromadb')
print(f'Vector store contains {store.count()} documents')
assert store.count() > 0, 'Run Udaplay_01_solution_project.ipynb first to build the index'


## 3. Construct the agent

`UdaPlayAgent` owns three tools, the conversation memory, and the state
machine. `confidence_floor=0.55` means anything below 0.55 confidence
from the evaluator triggers a web-search fallback.


In [ ]:
agent = UdaPlayAgent(store=store, top_k=4, confidence_floor=0.55)
print('Tools registered:')
for tool in (agent.retrieval_tool, agent.evaluation_tool, agent.web_search_tool):
    print(f'  - {tool.name}: {tool.description}')


## 4. Example queries

Run several questions through the agent in sequence. Because the agent
remembers prior turns, the third query references context from the
first.


In [ ]:
QUERIES = [
    'Who developed FIFA 21?',
    'When was God of War Ragnarok released?',
    'What platform was Pokemon Red launched on?',
    'What is Rockstar Games working on right now?',
]


In [ ]:
def show_report(report):
    print('=' * 80)
    print(f'Q: {report.question}')
    print('-' * 80)
    print(f'Answer:        {report.answer}')
    print(f'Confidence:    {report.confidence:.2f}')
    print(f'Web fallback?: {report.used_web_search}')
    print('Citations:')
    for c in report.citations:
        print(f'  [{c.label}] {c.source}')
    print('Trace:')
    for step in report.trace:
        print(f"  -> {step['state']}: {step['detail']}")
    print()

for q in QUERIES:
    show_report(agent.ask(q))


## 5. Structured output

The same report is also available as JSON for downstream integrations
(dashboards, evaluation pipelines, etc.).


In [ ]:
report = agent.ask('Who developed The Witcher 3?')
print(agent.to_json(report))


## 6. Conversation memory

Across calls the agent keeps a short transcript so follow-up questions
have context.


In [ ]:
_ = agent.ask('Who published Elden Ring?')
_ = agent.ask('And which studio developed it?')
print(agent.memory.transcript())


## 7. Long-term memory from web search

When the agent falls back to the web (e.g. for the Rockstar question),
the results are persisted into the vector store under `kind=web_memory`
so a follow-up question can answer from the local index.


In [ ]:
web_memories = [row for row in store.peek(20) if row['metadata'].get('kind') == 'web_memory']
print(f'{len(web_memories)} web-memory rows persisted')
for m in web_memories[:3]:
    print('-', m['id'], '->', m['metadata'].get('title'))


## 8. Report

Each agent run above prints, for every query:

- the final answer with inline `[KB-i]` / `[WEB-i]` citations,
- a confidence score from the evaluator,
- a boolean indicating whether the web fallback was triggered,
- the citation list (source paths or URLs), and
- the full state-machine trace.

That trace is the agent's reasoning record, satisfying the rubric
requirement to surface tool usage and reasoning alongside the answer.
